# 🏛️ Debris Careers — NLP & Machine Learning Career Consultant
### College Project: Information Retrieval & Cosine Similarity Recommendation Engine
- **Core ML Framework:** scikit-learn (`TfidfVectorizer`, `cosine_similarity`) & pandas
- **Architecture:** 5-Stage Machine Learning Pipeline (NLP Extraction &rarr; Vectorization &rarr; Similarity Calculation &rarr; Ranking &rarr; Template Explanation)
- **Interaction Model:** Multi-turn Conversational Chatbot (tracks profile answers, subjects, activities, and career priorities)
- **Aesthetics:** Minimalist Monochrome Liquid Glass (`backdrop-filter: blur`) with Ambient Floating Constellation Canvas
- **Security:** Email Authentication with strict **Maximum 8-Character Password Protection**
- **Offline Capability:** 100% Local execution — Zero external LLMs, Zero paid APIs, Zero network requirements


## 1. Database Initialization & Authentication Module
Creates SQLite tables for `users`, `conversations`, and `messages`.
Enforces the security constraint: **Password must be maximum 8 characters**.

In [ ]:
import sqlite3
import hashlib
import uuid
import json
import os

DB_NAME = 'debris_careers.db'

def get_db():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def hash_password(password: str) -> str:
    salt = 'debris_careers_salt_'
    return hashlib.sha256((salt + password).encode('utf-8')).hexdigest()

def setup_database():
    conn = get_db()
    cursor = conn.cursor()
    
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    ''')
    
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS conversations (
        id TEXT PRIMARY KEY,
        user_id INTEGER NOT NULL,
        title TEXT NOT NULL,
        state_data TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (user_id) REFERENCES users (id)
    );
    ''')
    
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS messages (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        conversation_id TEXT NOT NULL,
        sender TEXT NOT NULL,
        content TEXT NOT NULL,
        options TEXT,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (conversation_id) REFERENCES conversations (id)
    );
    ''')
    conn.commit()
    conn.close()
    print('✅ Debris Careers Database initialized.')

def register_user(email: str, password: str):
    email = email.strip().lower()
    if not email or '@' not in email:
        return False, 'Invalid email format.'
    if len(password) < 1:
        return False, 'Password cannot be empty.'
    if len(password) > 8:
        return False, 'Constraint Failed: Password must be maximum 8 characters.'
    
    pwd_hash = hash_password(password)
    conn = get_db()
    cursor = conn.cursor()
    try:
        cursor.execute('INSERT INTO users (email, password_hash) VALUES (?, ?)', (email, pwd_hash))
        conn.commit()
        return True, f'User {email} registered successfully.'
    except sqlite3.IntegrityError:
        return False, 'Account already exists.'
    finally:
        conn.close()

setup_database()
print(register_user('demo@debriscareers.com', 'pass123'))
# Verify security boundary constraint rejection (> 8 chars)
print(register_user('test@debriscareers.com', 'longpassword999'))


## 2. NLP Pipeline & TF-IDF Vectorization & Cosine Similarity
Demonstrates the 5-Stage Machine Learning approach:
1. **Information Extraction:** Extract subjects, skills, and values from student query.
2. **Vectorization:** Convert student text and career corpus into numerical TF-IDF vectors.
3. **Similarity Calculation:** Compute mathematical Cosine Similarity.
4. **Ranking:** Sort similarity scores in descending order.
5. **Explanations:** Generate verified career recommendations with colleges and next action steps.

In [ ]:
from rule_engine import NLP_ENGINE, DF_CAREERS
import pandas as pd

print(f'Total Careers in Database: {len(DF_CAREERS)}')
print('\nFeature Matrix Shape (TF-IDF Careers):', NLP_ENGINE.career_matrix.shape)

# Example Student Input
sample_input = 'I like mathematics, computers, and solving logical problems. I want a high-paying job.'
print(f'\nProcessing Sample Student Input:\n"{sample_input}"\n')

# Stage 1: Extracted Information
extracted = NLP_ENGINE.extract_student_attributes(sample_input)
print('Step 1 (NLP Information Extraction):')
print('  • Interests Detected:', extracted['interests'])
print('  • Priorities Detected:', extracted['priorities'])

# Stage 2, 3 & 4: TF-IDF & Cosine Similarity Ranking
results = NLP_ENGINE.recommend(sample_input, top_k=4)
print('\nStep 3 & 4 (Cosine Similarity Ranking):')
for r in results:
    print(f"  Rank #{r['rank']}: {r['career']['title']} | Similarity Score: {r['score']} ({r['score_percent']}%)")


## 3. Interactive Multi-Turn Chatbot Test
Simulates a multi-turn conversation tracking student subjects, activities, and career values.

In [ ]:
from rule_engine import ConversationalCareerBot

def chat_turn(message, state):
    reply, next_state, opts = ConversationalCareerBot.process_chat_message(message, state)
    print('=' * 75)
    print(f'USER: {message}')
    print('-' * 75)
    print(f'DEBRIS BOT:\n{reply}')
    print('-' * 75)
    print(f'Quick Options: {opts}\n')
    return next_state

# Simulating 3-turn interactive conversation
s0 = {'stage': 'stage_ask_subjects', 'profile_answers': []}
s1 = chat_turn('Mathematics and computers', s0)
s2 = chat_turn('Coding and solving problems', s1)
s3 = chat_turn('High salary and job stability', s2)


## 4. Launch the Debris Careers Web Application
Runs the Flask server with the new NLP engine and liquid frosted glass interface at **http://127.0.0.1:5001**.

In [ ]:
import threading
import time
from app import app

def start_server():
    app.run(host='127.0.0.1', port=5001, debug=False, use_reloader=False)

t = threading.Thread(target=start_server, daemon=True)
t.start()
time.sleep(1.5)

print('\n🚀 Debris Careers Web Application is LIVE with NLP & Cosine Similarity!')
print('👉 Open your browser at: http://127.0.0.1:5001\n')


## 5. College Viva Presentation Notes (NLP + ML)
1. **Information Retrieval (IR) vs Generative AI:**
   - Rather than relying on black-box LLMs that hallucinate, we convert text to a high-dimensional vector space using **TF-IDF (Term Frequency-Inverse Document Frequency)**.
2. **Mathematical Formula for Cosine Similarity:**
   $$\text{Cosine Similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}}$$
3. **Multi-turn Chatbot vs Static Form:**
   - The conversation manager maintains profile state across steps (Subjects &rarr; Daily Activities &rarr; Priority Values) and executes the ML vectorizer dynamically when profile data is sufficient.
4. **Aesthetic & Security Integration:**
   - 8-character password constraint strictly enforced.
   - Soothing liquid glassmorphism with Porsche typography and animated harmonic constellation canvas.